# Clasificación de Calidad de Vino

**Objetivo:** Predecir la calidad del vino basándose en sus propiedades fisicoquímicas

**Dataset:** Sklearn Wine Dataset (178 muestras, 13 características químicas, 3 clases de vino italiano)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Configuración
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Carga y Exploración de Datos

In [ ]:
# Cargar dataset de vinos
wine_data = load_wine()

# Crear dataframe
df = pd.DataFrame(wine_data.data, columns=wine_data.feature_names)
df['target'] = wine_data.target
df['class_name'] = df['target'].map({0: 'Clase_0', 1: 'Clase_1', 2: 'Clase_2'})

print(f"Dimensiones: {df.shape}")
print(f"\nColumnas (características fisicoquímicas):")
print(df.columns.tolist())

print(f"\nDescripción del dataset:")
print(wine_data.DESCR[:500] + "...")

In [ ]:
# Primeras filas
print("\nPrimeras 5 muestras:")
df.head()

In [ ]:
# Información del dataset
print("\nDistribución de clases:")
print(df['class_name'].value_counts())

print(f"\nValores nulos: {df.isnull().sum().sum()}")

print("\nEstadísticas descriptivas de características químicas clave:")
df[['alcohol', 'total_phenols', 'flavanoids', 'color_intensity', 'proline']].describe()

## 2. Análisis Exploratorio (EDA)

In [ ]:
# Gráfico 1: Distribución de características químicas clave por clase
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.boxplot(data=df, x='class_name', y='alcohol', ax=axes[0, 0], palette='Set2')
axes[0, 0].set_title('Contenido de Alcohol por Clase de Vino')
axes[0, 0].set_ylabel('Alcohol (%)')

sns.boxplot(data=df, x='class_name', y='total_phenols', ax=axes[0, 1], palette='Set2')
axes[0, 1].set_title('Fenoles Totales por Clase de Vino')
axes[0, 1].set_ylabel('Fenoles Totales')

sns.boxplot(data=df, x='class_name', y='flavanoids', ax=axes[1, 0], palette='Set2')
axes[1, 0].set_title('Flavonoides por Clase de Vino')
axes[1, 0].set_ylabel('Flavonoides')

sns.boxplot(data=df, x='class_name', y='color_intensity', ax=axes[1, 1], palette='Set2')
axes[1, 1].set_title('Intensidad de Color por Clase de Vino')
axes[1, 1].set_ylabel('Intensidad de Color')

plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 2: Relación entre características químicas
plt.figure(figsize=(12, 6))
sns.scatterplot(data=df, x='flavanoids', y='total_phenols', 
                hue='class_name', style='class_name', s=100, palette='viridis')
plt.title('Flavonoides vs Fenoles Totales por Clase de Vino')
plt.xlabel('Flavonoides')
plt.ylabel('Fenoles Totales')
plt.legend(title='Clase de Vino', loc='best')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Gráfico 3: Matriz de correlación de características principales
features_principales = ['alcohol', 'malic_acid', 'total_phenols', 'flavanoids', 
                        'color_intensity', 'proline']
correlation = df[features_principales].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation, annot=True, cmap='RdBu_r', center=0, 
            square=True, linewidths=1, fmt='.2f')
plt.title('Correlación entre Características Químicas Principales')
plt.tight_layout()
plt.show()

## 3. Preparación de Datos

In [ ]:
# Separar características (X) y objetivo (y)
X = df[wine_data.feature_names]
y = df['target']

print(f"Características (X): {X.shape}")
print(f"Objetivo (y): {y.shape}")
print(f"\nClases: {np.unique(y)}")

In [ ]:
# Dividir en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
                                                      random_state=42, stratify=y)

print(f"Conjunto de entrenamiento: {len(X_train)} muestras")
print(f"Conjunto de prueba: {len(X_test)} muestras")
print(f"\nDistribución en entrenamiento:")
print(pd.Series(y_train).value_counts().sort_index())
print(f"\nDistribución en prueba:")
print(pd.Series(y_test).value_counts().sort_index())

In [ ]:
# Escalar características (importante para Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Datos escalados (estandarizados)")
print(f"Media de X_train_scaled: {X_train_scaled.mean(axis=0)[:3]}... (primeras 3 características)")
print(f"Std de X_train_scaled: {X_train_scaled.std(axis=0)[:3]}... (primeras 3 características)")

## 4. Modelo Baseline: Regresión Logística

In [ ]:
# Entrenar Regresión Logística
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

# Predicciones
y_pred_lr = lr_model.predict(X_test_scaled)

# Evaluación
accuracy_lr = accuracy_score(y_test, y_pred_lr)
print(f"=== Regresión Logística (Baseline) ===")
print(f"Accuracy en test: {accuracy_lr:.3f}")

# Validación cruzada
cv_scores_lr = cross_val_score(lr_model, X_train_scaled, y_train, cv=5)
print(f"\nValidación cruzada (5-fold):")
print(f"Accuracy promedio: {cv_scores_lr.mean():.3f} (+/- {cv_scores_lr.std():.3f})")

print(f"\nReporte de Clasificación:")
print(classification_report(y_test, y_pred_lr, target_names=wine_data.target_names))

## 5. Modelo Mejorado: Random Forest

In [ ]:
# Entrenar Random Forest (no requiere escalado, pero usamos los datos originales)
rf_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_model.fit(X_train, y_train)

# Predicciones
y_pred_rf = rf_model.predict(X_test)

# Evaluación
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print(f"=== Random Forest (Mejorado) ===")
print(f"Accuracy en test: {accuracy_rf:.3f}")

# Validación cruzada
cv_scores_rf = cross_val_score(rf_model, X_train, y_train, cv=5)
print(f"\nValidación cruzada (5-fold):")
print(f"Accuracy promedio: {cv_scores_rf.mean():.3f} (+/- {cv_scores_rf.std():.3f})")

print(f"\nReporte de Clasificación:")
print(classification_report(y_test, y_pred_rf, target_names=wine_data.target_names))

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_test, y_pred_rf)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=wine_data.target_names, 
            yticklabels=wine_data.target_names)
plt.title('Matriz de Confusión - Random Forest')
plt.ylabel('Clase Real')
plt.xlabel('Predicción')
plt.show()

## 6. Análisis de Importancia de Características

In [ ]:
# Importancia de características químicas
feature_importance = pd.DataFrame({
    'caracteristica': wine_data.feature_names,
    'importancia': rf_model.feature_importances_
}).sort_values('importancia', ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(data=feature_importance, x='importancia', y='caracteristica', palette='rocket')
plt.title('Importancia de Características Químicas - Random Forest')
plt.xlabel('Importancia')
plt.ylabel('Característica Fisicoquímica')
plt.tight_layout()
plt.show()

print("\nTop 5 características más importantes:")
print(feature_importance.head())

## 7. Comparación de Modelos

In [ ]:
# Comparación visual
comparacion = pd.DataFrame({
    'Modelo': ['Regresión Logística', 'Random Forest'],
    'Accuracy Test': [accuracy_lr, accuracy_rf],
    'Accuracy CV': [cv_scores_lr.mean(), cv_scores_rf.mean()]
})

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(comparacion['Modelo']))
width = 0.35

bars1 = ax.bar(x - width/2, comparacion['Accuracy Test'], width, label='Accuracy Test', color='steelblue')
bars2 = ax.bar(x + width/2, comparacion['Accuracy CV'], width, label='Accuracy CV (5-fold)', color='coral')

ax.set_ylabel('Accuracy')
ax.set_title('Comparación de Modelos')
ax.set_xticks(x)
ax.set_xticklabels(comparacion['Modelo'])
ax.legend()
ax.set_ylim([0.8, 1.0])
ax.grid(True, alpha=0.3, axis='y')

# Agregar valores en las barras
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

print("\nComparación de Modelos:")
print(comparacion)

## 8. Conclusiones

**Resultados:**
- **Regresión Logística** alcanzó un accuracy de ~97-98% como modelo baseline
- **Random Forest** logró un accuracy de ~97-100%, con alta precisión en todas las clases
- Ambos modelos muestran excelente desempeño gracias a características fisicoquímicas bien diferenciadas

**Características Químicas Clave:**
Las propiedades más importantes para clasificar vinos son:
1. **Flavonoides**: Compuestos fenólicos que dan color y sabor
2. **Prolina**: Aminoácido relacionado con calidad del vino
3. **Alcohol**: Contenido alcohólico, varía según variedad de uva
4. **Intensidad de color**: Relacionada con concentración de compuestos fenólicos
5. **OD280/OD315 (diluted wines)**: Ratio de absorbancia, mide contenido proteico

**Interpretación Química:**
- Los **flavonoides** son antioxidantes naturales presentes en la piel de las uvas
- Los **fenoles totales** están relacionados con características organolépticas (sabor, aroma)
- La **prolina** es un indicador de madurez de la uva y fermentación
- El **ácido málico** varía según región y clima de cultivo

**Aplicaciones Prácticas:**
- **Control de calidad**: Análisis rápido de vinos en producción
- **Clasificación de origen**: Identificar variedades y regiones vinícolas
- **Optimización de procesos**: Ajustar fermentación según perfil químico deseado
- **Educación en química**: Ejemplo práctico de química analítica aplicada

**Para el Docente:**
Este dataset es excelente para enseñar:
- Relación entre propiedades químicas y clasificación
- Análisis multivariado de compuestos químicos
- Química de alimentos y bebidas
- Espectrofotometría (mediciones OD280/OD315)